In [35]:
# Аугментации
from torchvision.transforms.v2 import RandomRotation, RandomPhotometricDistort, \
RandomHorizontalFlip, RandomVerticalFlip, Compose, Resize
from torchvision.transforms import ToTensor, Normalize

# Загрузка датасета
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader
import torch
import kagglehub
import os
import shutil

# Модель
from torchsummary import summary
from torchvision.models import efficientnet_v2_s
import torch.nn as nn

# Обучение
import torch.optim as optim
from tqdm import tqdm

# Валидация
from sklearn.metrics import classification_report

# Визуализация
from matplotlib import pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# 1. Загрузка и предобработка данных

## Установка датасета

Используемый датасет: https://www.kaggle.com/datasets/richardradli/ogyeiv2/data

In [7]:
# Установка последней версии
downloaded_path = kagglehub.dataset_download("richardradli/ogyeiv2")

Resuming download from 220200960 bytes (3255382740 bytes left)...
Resuming download to /home/evgeniy/.cache/kagglehub/datasets/richardradli/ogyeiv2/3.archive (220200960/3475583700) bytes left.


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3.24G/3.24G [12:04<00:00, 4.49MB/s]

Extracting files...


In [16]:
# Перемещаем в data/ogyeiv
dataset_path = os.path.join('data', 'raw', 'digikala')
dataset_path = os.path.join(os.getcwd(), dataset_path)

if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)

shutil.move(downloaded_path, dataset_path)
print(f"Датасет сохранён в: {dataset_path}")

Датасет сохранён в: /home/evgeniy/Документы/GitHub/YandexNN/sprint_3/data/raw/digikala


## Создание DataLoaders

In [20]:
source_path = os.path.join(dataset_path, 'ogyeiv', 'ogyeiv2', 'ogyeiv2')
target_path = os.path.join(dataset_path, '..', '..', 'processed', 'digikala')
os.makedirs(target_path, exist_ok=True)
print(f"Датасет будет сохранён по пути: {target_path}")

Датасет будет сохранён по пути: /home/evgeniy/Документы/GitHub/YandexNN/sprint_3/data/raw/digikala/../../processed/digikala


In [21]:
# Приведение датасета к формату ImageFolder
folders = ["train", "valid", "test"]

for folder in folders:
    src_images = os.path.join(source_path, folder, "images")
    src_labels = os.path.join(source_path, folder, "labels")
    target_folder = os.path.join(target_path, folder)
    os.makedirs(target_folder, exist_ok=True)

    image_files = [
        f for f in os.listdir(src_images)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(f"Processing {folder}: {len(image_files)} images")

    for img_name in tqdm(image_files):
        # Extract class name from filename
        # acc_long_600_mg_s_022.jpg -> acc_long_600_mg
        class_name = "_".join(img_name.split("_")[:-2])

        class_dir = os.path.join(target_folder, class_name)
        os.makedirs(class_dir, exist_ok=True)

        src_img_path = os.path.join(src_images, img_name)
        trg_img_path = os.path.join(class_dir, img_name)

        shutil.copy2(src_img_path, trg_img_path)

Processing train: 3136 images


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3136/3136 [00:00<00:00, 5118.16it/s]


Processing valid: 672 images


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 672/672 [00:00<00:00, 5165.81it/s]


Processing test: 672 images


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 672/672 [00:00<00:00, 5101.11it/s]


In [30]:
# Аугментации
train_transforms = Compose([
    Resize((384, 384)),  # Рекомендуется для efficientnet_v2_s
    ToTensor(),
    Normalize((0.5), (0.5)),
    RandomHorizontalFlip(p=0.2),
    RandomVerticalFlip(p=0.2),
    RandomRotation([-5, 5], fill=255.)
])

val_transforms = Compose([
    Resize((384, 384)),  # Рекомендуется для efficientnet_v2_s
    ToTensor(),
    Normalize((0.5), (0.5))
]) 

In [31]:
train_dataset = ImageFolder(os.path.join(target_path, 'train'), transform=train_transforms)
val_dataset = ImageFolder(os.path.join(target_path, 'valid'), transform=val_transforms)
test_dataset = ImageFolder(os.path.join(target_path, 'test'), transform=val_transforms)

In [62]:
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [63]:
print("Количество изображений в train:", len(train_dataset))
print("Количество изображений в val:", len(val_dataset))
print("Количество изображений в test:", len(test_dataset))
print("Список классов:", train_dataset.classes) 

Количество изображений в train: 3136
Количество изображений в val: 672
Количество изображений в test: 672
Список классов: ['acc_long_600_mg', 'advil_ultra_forte', 'akineton_2_mg', 'algoflex_forte_dolo_400_mg', 'algoflex_rapid_400_mg', 'algopyrin_500_mg', 'ambroxol_egis_30_mg', 'apranax_550_mg', 'aspirin_ultra_500_mg', 'atoris_20_mg', 'atorvastatin_teva_20_mg', 'betaloc_50_mg', 'bila_git', 'c_vitamin_teva_500_mg', 'calci_kid', 'cataflam_50_mg', 'cataflam_dolo_25_mg', 'cataflam_v_50_mg', 'cetirizin_10_mg', 'co_perineva_4_mg_1_25_mg', 'co_xeter_20_mg_10_mg', 'cold_fx', 'coldrex', 'concor_10_mg', 'concor_5_mg', 'condrosulf_800_mg', 'controloc_20_mg', 'covercard_plus_10_mg_2_5_mg_5_mg', 'coverex_4_mg', 'diclopram_75-mg_20-mg', 'donalgin_250_mg', 'dorithricin_mentol', 'doxazosin_hexal_4_mg', 'doxazosin_sandoz_uro_4_mg', 'dulodet_60_mg', 'dulsevia_60_mg', 'enterol_250_mg', 'escitil_10_mg', 'favipiravir_meditop_200_mg', 'frontin_0_25_mg', 'frontin_0_5_mg', 'furon_40_mg', 'ibumax_400_mg', 'inda

# 2. Объявление модели

In [64]:
# Загрузка модели
model = efficientnet_v2_s(weights='IMAGENET1K_V1')
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [65]:
# Изменение только классификатора (может, этого хватит)
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

model.classifier = nn.Linear(1280, len(train_dataset.classes), bias=True)
summary(model, input_size=(3, 384, 384), device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 24, 192, 192]             648
       BatchNorm2d-2         [-1, 24, 192, 192]              48
              SiLU-3         [-1, 24, 192, 192]               0
            Conv2d-4         [-1, 24, 192, 192]           5,184
       BatchNorm2d-5         [-1, 24, 192, 192]              48
              SiLU-6         [-1, 24, 192, 192]               0
   StochasticDepth-7         [-1, 24, 192, 192]               0
       FusedMBConv-8         [-1, 24, 192, 192]               0
            Conv2d-9         [-1, 24, 192, 192]           5,184
      BatchNorm2d-10         [-1, 24, 192, 192]              48
             SiLU-11         [-1, 24, 192, 192]               0
  StochasticDepth-12         [-1, 24, 192, 192]               0
      FusedMBConv-13         [-1, 24, 192, 192]               0
           Conv2d-14           [-1, 96,

In [66]:
model = model.to(device)

# 3. Обучение модели

In [67]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) 

In [68]:
def train_one_epoch(epoch_index, train_model):
    total_loss = 0.0          # сумма потерь за всю эпоху
    running_loss = 0.0        # для скользящего среднего (20 батчей)
    correct = 0
    total = 0

    progress_bar = tqdm(enumerate(train_loader), 
                        total=len(train_loader),
                        desc=f'Эпоха {epoch_index}',
                        unit='batch')
    
    for batch_index, data in progress_bar:
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = train_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # -- накопление статистики --
        batch_loss = loss.item()
        total_loss += batch_loss
        running_loss += batch_loss

        # точность: предсказанный класс = argmax
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # -- прогресс-бар: средний loss за последние 20 батчей --
        if batch_index % 5 == 4:
            last_loss = running_loss / 5.0
            progress_bar.set_postfix({'loss': f'{last_loss:.4f}'})
            running_loss = 0.0

    # средние показатели за всю эпоху
    avg_loss = total_loss / len(train_loader)
    avg_acc = correct / total
    return avg_loss, avg_acc

In [ ]:
EPOCHS = 30
best_vloss = 1e5

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(EPOCHS):
    print(f'Эпоха {epoch}')

    # --- Обучение ---
    avg_loss, avg_acc = train_one_epoch(epoch, model)
    train_losses.append(avg_loss)
    train_accuracies.append(avg_acc)
    
    # --- Валидация ---
    model.eval()
    running_vloss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for i, vdata in enumerate(val_loader):
            vinputs, vlabels = vdata
            vinputs = vinputs.to(device)
            vlabels = vlabels.to(device)
            
            voutputs = model(vinputs)
            vloss = criterion(voutputs, vlabels)
            running_vloss += vloss.item()
            
            # --- Подсчёт accuracy ---
            _, predicted = torch.max(voutputs.data, 1)
            total += vlabels.size(0)
            correct += (predicted == vlabels).sum().item()
    
    avg_vloss = running_vloss / (i + 1)
    val_losses.append(avg_vloss)
    
    val_acc = correct / total
    val_accuracies.append(val_acc)

    # --- Сохранение лучшей модели по loss ---
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = f'models/pills_classifier/pills_classifier_{epoch}.pt'
        torch.save(model.state_dict(), model_path)

    # --- Вывод метрик ---
    print(f'Train loss: {avg_loss:.4f}, Train acc: {avg_acc:.4f}')
    print(f'Val loss:   {avg_vloss:.4f}, Val acc:   {val_acc:.4f}\n')

Эпоха 0


Эпоха 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 49/49 [01:50<00:00,  2.25s/batch, loss=4.6477]


Train loss: 4.7109, Train acc: 0.0207
Val loss:   4.4533, Val acc:   0.1027

Эпоха 1


Эпоха 1: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 49/49 [01:46<00:00,  2.17s/batch, loss=4.3878]


Train loss: 4.4455, Train acc: 0.0839
Val loss:   4.1670, Val acc:   0.1354

Эпоха 2


Эпоха 2: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 49/49 [01:47<00:00,  2.20s/batch, loss=4.2123]


Train loss: 4.2352, Train acc: 0.1317
Val loss:   3.9219, Val acc:   0.1830

Эпоха 3


Эпоха 3: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 49/49 [01:46<00:00,  2.18s/batch, loss=3.9687]


Train loss: 4.0344, Train acc: 0.1818
Val loss:   3.7473, Val acc:   0.2024

Эпоха 4


Эпоха 4:  18%|██████████████████▏                                                                                | 9/49 [00:19<01:27,  2.18s/batch, loss=3.9168]

## Графики обучения

In [ ]:
save_dir = 'docs/digikala'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
plt.figure(figsize=(12, 5))

# --- График ошибок (loss) ---
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, EPOCHS+1), val_losses, label='Val Loss', marker='s')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Динамика ошибки')
plt.legend()
plt.grid(True)

# --- График точности (accuracy) ---
plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS+1), train_accuracies, label='Train Accuracy', marker='o')
plt.plot(range(1, EPOCHS+1), val_accuracies, label='Val Accuracy', marker='s')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy')
plt.title('Динамика точности')
plt.legend()
plt.grid(True)

plt.tight_layout()

# Сохраняем в файл
plt.savefig(os.path.join(save_dir, 'training_history.png'), dpi=150)
plt.show()

# 4. Оценка качества

In [ ]:
labels_predicted = []
labels_true = []

model.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        # argmax по всем примерам, так как torch.max возвращает два параметра
        # максимальные значения в выборке и позиции, на которых они находятся (argmax)
        _, predicted = torch.max(outputs, 1) # argmax по всем примерам
        labels_predicted.extend(predicted.cpu().numpy())
        labels_true.extend(labels.cpu().numpy()) 

In [ ]:
print(classification_report(labels_true, labels_predicted, target_names=dataset.classes)) 

# 5. Анализ результатов

- На каких 5 классах модель ошибается чаще всего?
- Почему модель может ошибаться на этих классах?
- На каких классах модель не совершает ошибок?
- Почему эти классы модель распознаёт безошибочно?
- Как можно улучшить точность классификатора?
- Как ещё можно проанализировать результаты и ошибки модели?